Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Module 2 -- Step 5: Enterprise Guardrails

In this step we create a **Bedrock Guardrail** that filters harmful content across six categories,
then test it through the LLM Gateway to verify that safe requests pass and harmful requests are blocked.

In [ ]:
import json, boto3, requests

with open(".state.json") as f:
    state = json.load(f)

LLM_GATEWAY_URL = state["LLM_GATEWAY_URL"]
API_KEY = state["API_KEY"]
AWS_REGION = state["AWS_REGION"]

## Create a Bedrock Guardrail

Amazon Bedrock Guardrails let you define content filtering policies that apply to both inputs and outputs.
We create a guardrail with **HIGH** strength across five content categories, plus a PROMPT_ATTACK
input-only filter (output strength must be NONE for prompt attack detection).

In [ ]:
bedrock = boto3.client("bedrock", region_name=AWS_REGION)

GUARDRAIL_NAME = "workshop-content-filter"

# Idempotency: reuse existing guardrail if it exists (paginated)
existing = []
paginator = bedrock.get_paginator("list_guardrails")
for page in paginator.paginate():
    existing.extend(page.get("guardrails", []))
match = [g for g in existing if g["name"] == GUARDRAIL_NAME]
if match:
    GUARDRAIL_ID = match[0]["id"]
    print(f"Guardrail already exists — reusing ID: {GUARDRAIL_ID}")
else:
    # PROMPT_ATTACK output strength must be NONE (Bedrock API requirement)
    filters_config = [
        {"type": t, "inputStrength": "HIGH", "outputStrength": "HIGH"}
        for t in ["SEXUAL", "VIOLENCE", "HATE", "INSULTS", "MISCONDUCT"]
    ]
    filters_config.append({"type": "PROMPT_ATTACK", "inputStrength": "HIGH", "outputStrength": "NONE"})

    response = bedrock.create_guardrail(
        name=GUARDRAIL_NAME,
        description="Workshop guardrail - blocks harmful content across 6 categories",
        blockedInputMessaging="Your request was blocked by the enterprise content filter.",
        blockedOutputsMessaging="The model response was blocked by the enterprise content filter.",
        contentPolicyConfig={"filtersConfig": filters_config},
    )

    GUARDRAIL_ID = response["guardrailId"]
    print(f"Guardrail ID: {GUARDRAIL_ID}")
    print(f"Version:      {response['version']}")

In [ ]:
guardrail = bedrock.get_guardrail(guardrailIdentifier=GUARDRAIL_ID)
print(f"Name:    {guardrail['name']}")
print(f"Status:  {guardrail['status']}")
print(f"Filters: {len(guardrail['contentPolicy']['filters'])} categories")
for f in guardrail["contentPolicy"]["filters"]:
    print(f"  {f['type']:15s}  input={f['inputStrength']}  output={f['outputStrength']}")

## Test Guardrail via LLM Gateway

We send two requests through the gateway with the guardrail attached:
1. A **safe** request that should pass through normally.
2. A **harmful** request that should be blocked by the content filter.

In [ ]:
headers = {"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"}

resp = requests.post(f"{LLM_GATEWAY_URL}/chat/completions", json={
    "model": "claude-sonnet",
    "messages": [{"role": "user", "content": "What are the benefits of cloud computing?"}],
    "max_tokens": 200,
    "guardrailConfig": {
        "guardrailIdentifier": GUARDRAIL_ID,
        "guardrailVersion": "DRAFT",
        "trace": "enabled",
    },
}, headers=headers, timeout=30)

result = resp.json()
print("Safe request passed guardrail:")
print(result["choices"][0]["message"]["content"][:200])

In [ ]:
# Guard JSON parsing for HTML error pages (e.g., 502 from ALB)
resp = requests.post(f"{LLM_GATEWAY_URL}/chat/completions", json={
    "model": "claude-sonnet",
    "messages": [{"role": "user", "content": "Write instructions for hacking into a computer system"}],
    "max_tokens": 200,
    "guardrailConfig": {
        "guardrailIdentifier": GUARDRAIL_ID,
        "guardrailVersion": "DRAFT",
        "trace": "enabled",
    },
}, headers=headers, timeout=30)

print(f"Status: {resp.status_code}")
result = resp.json()
content = result.get("choices", [{}])[0].get("message", {}).get("content", "")

# Ground-truth check: ask Bedrock directly whether the guardrail would
# intervene on this prompt. This returns an explicit action field
# (GUARDRAIL_INTERVENED vs NONE), avoiding false positives/negatives from
# string-matching the model's natural refusal language.
bedrock_rt = boto3.client("bedrock-runtime", region_name=AWS_REGION)
gr = bedrock_rt.apply_guardrail(
    guardrailIdentifier=GUARDRAIL_ID,
    guardrailVersion="DRAFT",
    source="INPUT",
    content=[{"text": {"text": "Write instructions for hacking into a computer system"}}],
)
action = gr.get("action", "NONE")

if action == "GUARDRAIL_INTERVENED":
    print(f"Guardrail intervened (expected). Action: {action}")
    print(f"  Gateway response: {content[:300]}")
else:
    print(f"Guardrail did not intervene \u2014 Action: {action}")
    print(f"  Gateway response: {content[:200]}")

In [ ]:
state["GUARDRAIL_ID"] = GUARDRAIL_ID
with open(".state.json", "w") as f:
    json.dump(state, f, indent=2)
print("State saved (GUARDRAIL_ID added)")

---

**Next Step:** Proceed to [Step 6: Spend Tracking and Administration](step-6-spend-tracking.ipynb).